# Import de PySpark et des clés

In [1]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
from datetime import datetime, date

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [2]:
import pandas as pd 
mat_inf = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_INFORMATIQUE/MATERIEL_INFORMATIQUE_20260429.txt',sep=';').columns
personnel = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt',sep=';').columns
mission = pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_MISSION/MISSION_20260429.txt',sep=';').columns
print(pd.read_csv('BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt',sep=';')['TS_CREATION_PERSONNEL'])
mat_inf = list(mat_inf)
personnel = list(personnel)
mission = list(mission)

KeyPersoMission = []
for i in personnel :
    if i in mission : 
        KeyPersoMission.append(i)

KeyPersoMission

0       2000-05-09 11:54:25
1       2014-08-17 01:27:04
2       2013-01-22 14:11:50
3       2007-08-27 01:28:18
4       2012-03-02 18:24:08
               ...         
4238    2002-07-10 13:02:23
4239    2023-06-25 19:12:37
4240    2020-06-27 04:11:52
4241    2007-06-06 12:15:27
4242    2003-10-02 11:51:43
Name: TS_CREATION_PERSONNEL, Length: 4243, dtype: object


['ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL']

In [4]:
print(mat_inf)
print(personnel)
print(mission)

['ID_MATERIELINFO', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DATE_ACHAT', 'TYPE', 'MODELE']
['ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DT_NAISS', 'VILLE_NAISS', 'PAYS_NAISS', 'NUM_SECU', 'IND_PAYS_NUM_TELP', 'NUM_TELEPHONE', 'NUM_VOIE', 'DSC_VOIE', 'CMPL_VOIE', 'CD_POSTAL', 'VILLE', 'PAYS', 'FONCTION_PERSONNEL', 'TS_CREATION_PERSONNEL', 'TS_MAJ_PPERSONNEL']
['ID_MISSION', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL', 'DATE_MISSION', 'TYPE_MISSION', 'VILLE_DEPART', 'PAYS_DEPART', 'VILLE_DESTINATION', 'PAYS_DESTINATION', 'TRANSPORT', 'ALLER_RETOUR']


# Tables
- FAITS MISSION
    - ID_PERSONNEL
    - ID_MISSION
    - ANNEE
    - MOIS
    - JOUR
    - SITE
- FAITS MATERIEL INFORMATIQUE
    - ID_PERSONNEL
    - ID_MATERIELINFO
    - ANNEE
    - MOIS
    - JOUR
    - SITE

- DATE : ANNEE, MOIS, JOUR
    - (From DATE_MISSION/DATE_ACHAT)
        - ANNEE
        - MOIS
        - JOUR
- SITE : SITE
    - SITE (From PERSONNEL.VILLE)
- MISSION : ID_MISSION
    - (from MISSION) : ID_MISSION, VILLE_DEPART, VILLE_DESTINATION,TYPE_MISSION,TRANSPORT,ALLER_RETOUR
    - (From colonne calculée) : IMPACT CARBONE
- LOCALISATION : Ville
    - (From MISSION.VILLE_DEPART, MISSION.VILLE_DESTINATION) : VILLE
    - (From MISSION.PAYS_DEPART,MISSION.PAYS_DESTINATION) : PAYS
    - (From : colonne calculée (geopy ?)) : CONTINENT
- Personnel - ID_PERSONNEL
    - ID_PERSONNEL, PAYS, FONCTION_PERSONNEL, DT_NAISSANCE
- MATERIEL INFORMATIQUE - ID_MATERIELINFO
    - Matériel informatique : ID_MATERIELINFO, DATE_ACHAT, TYPE (Split et calculé), Ecran (calculé), MODELE
    - Materiel informatique impact : Impact (join on type and modèle)

In [2]:
import glob
#Charger les fichiers de personnel
fichiers_personnel = glob.glob('BDD_BGES/**/PERSONNEL_*.txt', recursive=True)
psdf_personnel_list = [ps.read_csv(f, sep=';',index_col='ID_PERSONNEL') for f in fichiers_personnel]
psdf_personnel = ps.concat(psdf_personnel_list, ignore_index=False)

#Charger les fichiers de matériel informatique
fichiers_mat_inf = glob.glob('BDD_BGES/**/MATERIEL_INFORMATIQUE_*.txt', recursive=True)
psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
psdf_mat_inf_temp = ps.concat(psdf_mat_inf_list, ignore_index=True)

#Charge les données d'impact carbone du matériel informatique
psdf_impact_matinf = ps.read_csv('BDD_BGES/materiel_informatique_impact.csv',sep=',')
psdf_impact_matinf = psdf_impact_matinf.rename(columns={
    'Type': 'TYPE',
    'Impact': 'IMPACT',
    'Modèle': 'MODELE'
})

#Fusionne les données de matériel informatique avec les données d'impact carbone
psdf_mat_inf = psdf_mat_inf_temp.merge(
    psdf_impact_matinf,
    on=['TYPE','MODELE'],
    how='inner'
)
#Impact est en kg/CO2, il faut le convertir en tonnes de CO2
psdf_mat_inf['IMPACT'] = psdf_mat_inf['IMPACT'] / 1000
psdf_mat_inf = psdf_mat_inf.set_index('ID_MATERIELINFO')

#Charger les fichiers de mission
fichiers_mission = glob.glob('BDD_BGES/**/MISSION_*.txt', recursive=True)
psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)

c:\Users\rzong\anaconda3\envs\nf26_env\Lib\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [5]:
# Transformation en spark dataframe
from pyspark.sql.functions import *
sdf_personnel = psdf_personnel.to_spark(index_col='ID_PERSONNEL')
sdf_personnel.show(10)
sdf_mat_inf = psdf_mat_inf.to_spark(index_col='ID_MATERIELINFO')
sdf_mat_inf.show(10)
sdf_mission = psdf_mission.to_spark(index_col='ID_MISSION')
sdf_mission.show(10)

+--------------------+-------------+----------------+----------+------------+----------+-----------+-----------------+-------------+--------+----------+---------+---------+------+-------+------------------+---------------------+-------------------+
|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|  DT_NAISS| VILLE_NAISS|PAYS_NAISS|   NUM_SECU|IND_PAYS_NUM_TELP|NUM_TELEPHONE|NUM_VOIE|  DSC_VOIE|CMPL_VOIE|CD_POSTAL| VILLE|   PAYS|FONCTION_PERSONNEL|TS_CREATION_PERSONNEL|  TS_MAJ_PPERSONNEL|
+--------------------+-------------+----------------+----------+------------+----------+-----------+-----------------+-------------+--------+----------+---------+---------+------+-------+------------------+---------------------+-------------------+
|KeyPers_Berlin_12...|        Name0|       FistName0|1948-11-17|      Bogota|  Colombia|NS000000000|             NULL|+336##0530601|      51|NomVoie520|     NULL|    #8356|Berlin|Germany|    Dateningenieur|  2000-05-09 11:54:25|2000-05-09 11:54:25|
|Key

In [7]:
DATE = sdf_mission.select(col('DATE_MISSION').alias('DATE')).join(sdf_mat_inf.select(col('DATE_ACHAT').alias('DATE')), on='DATE', how='outer')
DATE = DATE.select(to_date(col('DATE'), 'yyyy-MM-dd').alias('DATE'))
DATE = DATE.select(year(col('DATE')).alias('ANNEE'), month(col('DATE')).alias('MOIS'), dayofmonth(col('DATE')).alias('JOUR'))
DATE.show(10)

+-----+----+----+
|ANNEE|MOIS|JOUR|
+-----+----+----+
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
| 2026|   4|  29|
+-----+----+----+
only showing top 10 rows


In [8]:
#Création du modèle étoile
FAITS_MISSION = sdf_mission.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select('ID_MISSION','ID_PERSONNEL',col('VILLE').alias('SITE'),year(to_date(col('DATE_MISSION'))).alias('ANNEE'), month(to_date(col('DATE_MISSION'))).alias('MOIS'), dayofmonth(to_date(col('DATE_MISSION'))).alias('JOUR'))
FAITS_MISSION.show(10)

SITE = sdf_personnel.select(col('VILLE').alias('SITE')).distinct()
SITE.show(10)

+----------------+--------------------+------+-----+----+----+
|      ID_MISSION|        ID_PERSONNEL|  SITE|ANNEE|MOIS|JOUR|
+----------------+--------------------+------+-----+----+----+
|BERLIN_202604290|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604291|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604292|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604293|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604294|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604295|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604296|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604297|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604298|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
|BERLIN_202604299|KeyPers_Berlin_12...|Berlin| 2026|   4|  29|
+----------------+--------------------+------+-----+----+----+
only showing top 10 rows
+-----------+
|       SITE|
+-----------+
|     Berlin|
|     London|
|Los Angeles|
|   New-Y

In [19]:
from geopy.distance import geodesic
from geopy.geocoders import Nominatim
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StringType

geolocator = Nominatim(user_agent="geo_app")


def distance_km(ville_dep, ville_arr, pays_dep=None, pays_arr=None):
    # Construction des requêtes
    query_dep = f"{ville_dep}, {pays_dep}" if pays_dep else ville_dep
    query_arr = f"{ville_arr}, {pays_arr}" if pays_arr else ville_arr

    # Géocodage
    loc_dep = geolocator.geocode(query_dep)
    loc_arr = geolocator.geocode(query_arr)

    # Sécurité
    if loc_dep is None or loc_arr is None:
        raise ValueError(f"Ville non trouvée : {query_dep} ou {query_arr}")

    coords_dep = (loc_dep.latitude, loc_dep.longitude)
    coords_arr = (loc_arr.latitude, loc_arr.longitude)

    return geodesic(coords_dep, coords_arr).km



# UDF pour le calcul d'impact carbone
def calcul_impact_carbone(distance_km, mode_transport):
    #retour en tonnes équivalent CO2
    d = distance_km
    mode = mode_transport.lower() if mode_transport else ""

    if mode == "avion":
        if d < 1000: #court courrier
            return d * 0.000258
        elif d < 3500: #moyen courrier
            return d * 0.000187
        else: #long courrier
            return d * 0.000152

    elif mode == "train":
        return d * 0.000004

    elif mode == "taxi":
        return d * 0.000192

    elif mode == "transports en commun":
        return d * 0.00005

    else:
        return 0.0

# Enregistrement de l'UDF
calcul_impact_carbone_udf = udf(calcul_impact_carbone, DoubleType())

# Étape 1: Collecter les villes uniques et géocoder en pandas
villes_df = sdf_mission.select('VILLE_DEPART', 'PAYS_DEPART').distinct().toPandas()
villes_dest_df = sdf_mission.select('VILLE_DESTINATION', 'PAYS_DESTINATION').distinct().toPandas()
villes_dest_df.columns = ['VILLE_DEPART', 'PAYS_DEPART']
all_villes = pd.concat([villes_df, villes_dest_df]).drop_duplicates()

# Géocodage des villes avec la fonction distance_km
import time

coord_dict = {}
for _, row in all_villes.iterrows():
    ville = row['VILLE_DEPART']
    pays = row['PAYS_DEPART']
    if pd.notna(ville) and pd.notna(pays) and (ville, pays) not in coord_dict:
        try:
            # Géocoder chaque ville individuellement
            query = f"{ville}, {pays}"
            loc = geolocator.geocode(query)
            if loc:
                coord_dict[(ville, pays)] = (loc.latitude, loc.longitude)
                time.sleep(1)  # Délai pour éviter la limitation de taux
        except Exception as e:
            print(f"Erreur géocodage {ville}, {pays}: {e}")

print(f"Nombre de villes géocodées: {len(coord_dict)}")
print(f"Exemples de coordonnées: {list(coord_dict.items())[:5]}")

# Étape 2: Convertir en pandas et calculer les distances
pdf_mission = sdf_mission.toPandas()

# Fonction pour calculer la distance en utilisant distance_km
def calc_distance(row):
    try:
        coords_dep = coord_dict.get((row['VILLE_DEPART'], row['PAYS_DEPART']))
        coords_arr = coord_dict.get((row['VILLE_DESTINATION'], row['PAYS_DESTINATION']))
        if coords_dep and coords_arr:
            return geodesic(coords_dep, coords_arr).km
    except:
        pass
    return 0.0

pdf_mission['DISTANCE_KM'] = pdf_mission.apply(calc_distance, axis=1)

# Débogage: afficher les distances
print(f"Distances non nulles: {(pdf_mission['DISTANCE_KM'] > 0).sum()}")
print(f"Exemples de distances:\n{pdf_mission[['VILLE_DEPART', 'VILLE_DESTINATION', 'DISTANCE_KM']].head(10)}")

# Fonction pour calculer l'impact carbone
def calc_impact(row):
    d = row['DISTANCE_KM']
    mode = str(row['TRANSPORT']).lower() if pd.notna(row['TRANSPORT']) else ""
    
    if mode == "avion":
        if d < 1000:
            return d * 0.000258
        elif d < 3500:
            return d * 0.000187
        else:
            return d * 0.000152
    elif mode == "train":
        return d * 0.000004
    elif mode == "taxi":
        return d * 0.000192
    elif mode == "transports en commun":
        return d * 0.00005
    return 0.0

pdf_mission['IMPACT_CARBONE'] = pdf_mission.apply(calc_impact, axis=1)

# Garder en pandas (plus stable avec numpy arrays)
MISSION = pdf_mission.copy()
MISSION.head(10)

Erreur géocodage Shanghai, China: Service timed out
Erreur géocodage Berlin, Allemagne: Service timed out
Erreur géocodage Paris, France: Service timed out
Erreur géocodage New-York, USA: Service timed out
Erreur géocodage Los Angeles, USA: Service timed out
Erreur géocodage London, England: Service timed out
Erreur géocodage Dubaï, Emirats: Service timed out
Erreur géocodage Wellington, New Zealand: Service timed out
Erreur géocodage Helsinki, Finlande: Service timed out
Erreur géocodage Vancouver, Canada: Service timed out
Erreur géocodage Stockholm, Suède: Service timed out
Erreur géocodage Compiègne, France: Service timed out
Erreur géocodage Alger, Algeria: Service timed out
Erreur géocodage Montreal, Canada: Service timed out
Erreur géocodage Pekin, China: Service timed out
Erreur géocodage Washington, USA: Service timed out
Erreur géocodage Marseille, France: Service timed out
Erreur géocodage Bordeaux, France: Service timed out
Erreur géocodage Melbourne, Australia: Service tim

,ID_MISSION,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_MISSION,TYPE_MISSION,VILLE_DEPART,PAYS_DEPART,VILLE_DESTINATION,PAYS_DESTINATION,TRANSPORT,ALLER_RETOUR,DISTANCE_KM,IMPACT_CARBONE
0,BERLIN_202604290,KeyPers_Berlin_1230422,Name422,FistName422,2026-04-29 11:36:14,Meeting,Berlin,Allemagne,Washington,USA,Avion,oui,0.0,0.0
1,BERLIN_202604291,KeyPers_Berlin_1231765,Name1765,FistName1765,2026-04-29 01:52:28,Schulung,Berlin,Allemagne,Alger,Algeria,Avion,oui,0.0,0.0
2,BERLIN_202604292,KeyPers_Berlin_1230939,Name939,FistName939,2026-04-29 05:26:50,Konferenz,Berlin,Allemagne,London,England,Avion,oui,0.0,0.0
3,BERLIN_202604293,KeyPers_Berlin_1233684,Name3684,FistName3684,2026-04-29 16:52:20,Entwicklung,Berlin,Allemagne,Montreal,Canada,Avion,oui,0.0,0.0
4,BERLIN_202604294,KeyPers_Berlin_1231377,Name1377,FistName1377,2026-04-29 09:39:49,Entwicklung,Shanghai,China,New-York,USA,Avion,oui,0.0,0.0
5,BERLIN_202604295,KeyPers_Berlin_1234088,Name4088,FistName4088,2026-04-29 15:01:21,Konferenz,Berlin,Allemagne,Berlin,Allemagne,Transports en commun,oui,0.0,0.0
6,BERLIN_202604296,KeyPers_Berlin_1231851,Name1851,FistName1851,2026-04-29 09:05:37,Entwicklung,Berlin,Allemagne,Bordeaux,France,Avion,oui,0.0,0.0
7,BERLIN_202604297,KeyPers_Berlin_1230522,Name522,FistName522,2026-04-29 17:51:50,Meeting,Berlin,Allemagne,Helsinki,Finlande,Avion,oui,0.0,0.0
8,BERLIN_202604298,KeyPers_Berlin_1231990,Name1990,FistName1990,2026-04-29 14:10:17,Meeting,Berlin,Allemagne,Montreal,Canada,Avion,oui,0.0,0.0
9,BERLIN_202604299,KeyPers_Berlin_1233372,Name3372,FistName3372,2026-04-29 18:01:23,Schulung,Berlin,Allemagne,Vancouver,Canada,Avion,oui,0.0,0.0


In [ ]:
#Création de la table LOCALISATION
from pyspark.sql.types import StringType
import country_converter as coco
from pyspark.sql.functions import udf
from deep_translator import GoogleTranslator

cc = coco.CountryConverter()

def get_continent(pays):
    try:
        continent = cc.convert(names=pays, to='continent')
        if continent == "not found":
            return None
        return continent
    except:
        return None
    

LOCALISATION = sdf_mission.select(
    col("VILLE_DEPART").alias("VILLE"),
    col("PAYS_DEPART").alias("PAYS")
).union(
    sdf_mission.select(
        col("VILLE_DESTINATION").alias("VILLE"),
        col("PAYS_DESTINATION").alias("PAYS")
    )
).distinct()

rows = LOCALISATION.collect()
pays_uniques = [row["PAYS"] for row in LOCALISATION.select("PAYS").distinct().collect()]
translator = GoogleTranslator(source='auto', target='en')

dict_pays_en = {
    pays: ("Sweden" if pays.lower() in ["suede", "suède"] else translator.translate(pays))
    for pays in pays_uniques
}

dict_pays_continent = {
    pays_fr: get_continent(dict_pays_en[pays_fr])
    for pays_fr in dict_pays_en
}

mapping_list = []
for k, v in dict_pays_continent.items():
    mapping_list += [lit(k), lit(v)]

mapping_expr = create_map(*mapping_list)
LOCALISATION = LOCALISATION.withColumn(
    "CONTINENT",
    mapping_expr[col("PAYS")]
)
LOCALISATION.show(10)

Suede not found in regex


In [9]:
FAITS_MATERIEL_INFORMATIQUE = sdf_mat_inf.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select('ID_PERSONNEL', 'ID_MATERIELINFO', year(to_date(col('DATE_ACHAT'))).alias('ANNEE'), month(to_date(col('DATE_ACHAT'))).alias('MOIS'), dayofmonth(to_date(col('DATE_ACHAT'))).alias('JOUR'), col('VILLE').alias('SITE'))
FAITS_MATERIEL_INFORMATIQUE.show(10)

+--------------------+--------------------+-----+----+----+------+
|        ID_PERSONNEL|     ID_MATERIELINFO|ANNEE|MOIS|JOUR|  SITE|
+--------------------+--------------------+-----+----+----+------+
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  29|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  29|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  29|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  30|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  30|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   4|  30|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   1|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   2|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   2|Berlin|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...| 2026|   5|   2|Berlin|
+--------------------+--------------------+-----+----+----+------+
only showing top 10 rows


In [16]:
PERSONNEL = sdf_personnel.select('ID_PERSONNEL','PAYS','FONCTION_PERSONNEL',col('DT_NAISS').alias('DATE_NAISSANCE'))
PERSONNEL.show(10)

+--------------------+-------+------------------+--------------+
|        ID_PERSONNEL|   PAYS|FONCTION_PERSONNEL|DATE_NAISSANCE|
+--------------------+-------+------------------+--------------+
|KeyPers_Berlin_12...|Germany|    Dateningenieur|    1948-11-17|
|KeyPers_Berlin_12...|Germany|     Führungskraft|    1941-03-05|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1958-09-14|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1961-04-18|
|KeyPers_Berlin_12...|Germany|    Dateningenieur|    1950-09-08|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1977-09-09|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    2000-10-08|
|KeyPers_Berlin_12...|Germany|     Führungskraft|    1997-04-08|
|KeyPers_Berlin_12...|Germany|    Dateningenieur|    1933-04-07|
|KeyPers_Berlin_12...|Germany| Computeringenieur|    1975-03-12|
+--------------------+-------+------------------+--------------+
only showing top 10 rows


In [11]:
MATERIEL_INFORMATIQUE = sdf_mat_inf.select('ID_MATERIELINFO', 'TYPE', 'MODELE', 'IMPACT')
MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
    "ECRAN",
    when(lower(col("TYPE")).contains("sans ecran"), "non")
    .otherwise("oui")
).withColumn(
    "TYPE_CLEAN",
    trim(split(lower(col("TYPE")), "sans ecran").getItem(0))
).select('ID_MATERIELINFO', col('TYPE_CLEAN').alias('TYPE'), 'MODELE', 'IMPACT', 'ECRAN')
MATERIEL_INFORMATIQUE.show(10)

+--------------------+----------------+--------------------+------+-----+
|     ID_MATERIELINFO|            TYPE|              MODELE|IMPACT|ECRAN|
+--------------------+----------------+--------------------+------+-----+
|BERLIN_MATERIEL_I...|         pc fixe|Precision tower 7xxx| 0.555|  non|
|BERLIN_MATERIEL_I...|     pc portable|       EliteBook 6xx| 0.175|  oui|
|BERLIN_MATERIEL_I...|vidéo projecteur|   modèle par défaut| 0.094|  oui|
|BERLIN_MATERIEL_I...|     pc portable|         MacBook air|  0.25|  oui|
|BERLIN_MATERIEL_I...|     pc portable|      EliteBook 10xx| 0.223|  oui|
|BERLIN_MATERIEL_I...|         pc fixe|EliteDesk (Tower,...| 0.393|  non|
|BERLIN_MATERIEL_I...|           ecran|    32pouces et plus|  0.59|  oui|
|BERLIN_MATERIEL_I...|      imprimante|   modèle par défaut|   0.5|  oui|
|BERLIN_MATERIEL_I...|         pc fixe|Precision tower 3xxx| 0.285|  non|
|BERLIN_MATERIEL_I...|     pc portable|       Latitude 5xxx|   0.3|  oui|
+--------------------+----------------

{'Geschäftstreffen': 'Business Meeting',
 'Konferenz': 'Conference',
 'Schulung': 'Training',
 'Meeting': 'Internal Meeting',
 'Entwicklung': 'Development',
 'Business Meeting': 'Business Meeting',
 'Conference': 'Conference',
 'Team Meeting': 'Internal Meeting',
 'Development': 'Development',
 'Vocational Training': 'Training',
 'Conférence': 'Conference',
 'Formation': 'Training',
 'Réunion': 'Internal Meeting',
 'Rencontre entreprises': 'Business Meeting',
 'Développement': 'Development'}

In [12]:
from pyspark.sql.functions import col, count, when, current_date, trim

# Fonction pour compter les valeurs nulles
def count_nulls(df, name):
    print(f"Null values dans {name}:")
    df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)
    print("\n")

# Helper pour détecter les chaînes vides ou nulles
def is_blank(column):
    return column.isNull() | (trim(column) == "")

# Fonction pour afficher les lignes suspectes selon un filtre
def show_suspect_rows(df, name, condition, message):
    print(f"{name} - {message}")
    df.filter(condition).show(20, truncate=False)
    print("\n")

# Comptage global des valeurs manquantes
count_nulls(Faits, "Faits")
count_nulls(Trajets, "Trajets")
count_nulls(Personnel, "Personnel")
count_nulls(MaterielInformatique, "MaterielInformatique")

# Recherche des valeurs aberrantes ou incohérences dans les tables dérivées
show_suspect_rows(
    Faits,
    "Faits",
    col("ID_MISSION").isNull() | col("ID_PERSONNEL").isNull() | col("ID_MATERIELINFO").isNull(),
    "lignes avec clés manquantes"
)

show_suspect_rows(
    Trajets,
    "Trajets",
    is_blank(col("PAYS_DEPART")) |
    is_blank(col("PAYS_DESTINATION")) |
    is_blank(col("VILLE_DEPART")) |
    is_blank(col("VILLE_DESTINATION")),
    "trajets avec pays ou villes manquants"
)

show_suspect_rows(
    Trajets,
    "Trajets",
    col("VILLE_DEPART") == col("VILLE_DESTINATION"),
    "trajets où la ville de départ est identique à la ville de destination"
)

show_suspect_rows(
    Personnel,
    "Personnel",
    is_blank(col("VILLE")) |
    is_blank(col("PAYS")) |
    is_blank(col("FONCTION_PERSONNEL")) |
    (col("TS_CREATION_PERSONNEL") > current_date()) |
    (col("TS_MAJ_PPERSONNEL") > current_date()),
    "personnel avec informations manquantes ou dates futures"
)

show_suspect_rows(
    MaterielInformatique,
    "MaterielInformatique",
    is_blank(col("TYPE")) |
    is_blank(col("MODELE")) |
    col("IMPACT").isNull() |
    (col("IMPACT") <= 0) |
    (col("DATE_ACHAT") > current_date()),
    "matériel avec type/modèle manquant, impact incorrect ou date d'achat future"
)

NameError: name 'Faits' is not defined

## ETL : Normalisation des données
#### A faire : 
- Tout traduire en anglais (exemple la fonction de directeur à berlin s'appelle Führungskraft)

## Requête sur le DataWarehouse

#### 1. Combien de cadres travaillent sur le site de Paris?

#### 2. Combien d’ingénieurs Data travaillent sur les sites aux États-Unis ?

#### 3. Combien d’ingénieurs informaticiens travaillent dans l’organisation (tous sites compris) ?

#### 4. Combien de PC fixes ont été achetés par l’organisation entre juin et septembre 2026 ?

#### 5. Quelle a été l’impact carbone des PC fixes sans ecran entre mai et octobre 2026 ?

#### 6. Quel a été l’impact carbone des PC portables achetés par les ingénieurs Data entre mai et octobre 2026 sur les sites de Londres et New-York?

#### 7. Quel a été l’impact carbone des écrans achetés par les cadres entre juillet et septembre 2026 sur tous les sites de l’organisation ?

#### 8. Quel a été l’impact carbone des missions sur les sites Européens entre mai et octobre 2026 ?

#### 9. Quels ont été les 5 jours les plus impactants concernant les missions en avion pour les sites Européens de l’organisation ?

#### 10. Quel a été le secteur d’activité qui a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

#### 11.  Quel site a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

#### 12. Quel a été l’impact carbone des missions reliant chaque site (la ville de départ est un site de l’organisation et la ville d’arrivée également) durant le mois de septembre 2026 ?

#### 13. Quel a été l’impact carbone des séminaires en juillet 2026 pour les employés de Los Angeles ?

#### 14. Quel secteur d’activité a été le plus impactant pour les missions “conférences” entre mai et septembre 2026 ?

#### 15. Quel a été l’âge moyen des employés Ingénieurs Data qui sont partis en formations entre juillet et septembre 2026 ?

#### 16. Quelle destination a été la plus impactante (en cumul) entre mai et octobre 2026 ?

#### 17. Quelles ont été les trois catégories de missions les plus impactantes pour les cadres dans les sites Européens en mai 2026 ?

*Les réponses des questions suivantes devront être sous forme de figures*
#### 18. Quelles ont été les 5 missions les plus impactantes sur le site de Paris ?

#### 19. Proposer une figure comparant l’impact carbone mensuel des missions en fonction du type de transport et sur chaque site.

#### 20. Proposer une figure illustrant l’impact carbone global mensuel de l’organisation.